# 🧠 3. Protein Embeddings Generation

Este notebook es el núcleo computacional del pipeline de **ProtT5**. En esta etapa, las secuencias de aminoácidos se transforman en vectores numéricos densos (embeddings) que capturan las propiedades biológicas y estructurales de las proteínas.

---

## 📋 Descripción General
El script utiliza modelos de lenguaje pre-entrenados para convertir las secuencias de texto en representaciones de **768 dimensiones**. Este proceso es fundamental para que los modelos de Deep Learning puedan "entender" la relación entre diferentes aminoácidos en una secuencia.



## 🛠️ Especificaciones Técnicas

1.  **Modelo y Arquitectura:**
    * Basado en modelos de lenguaje específicos para proteínas (ProtBert/ProtFlash).
    * Utiliza un método de **Mean Pooling** sobre los tokens para obtener una representación única por cada proteína.
2.  **Configuración de Hardware:**
    * Optimizado para ejecutarse en **GPU (T4)** mediante el uso de tensores y aceleración CUDA.
3.  **Procesamiento Eficiente:**
    * Implementa **Batch Processing** (tamaño de lote de 64) para maximizar el uso de la memoria de video y acelerar la inferencia.
4.  **Flujo de Trabajo:**
    * Carga los archivos `train.csv`, `val.csv` y `test.csv` generados anteriormente.
    * Genera embeddings para un total de **40,358 secuencias**.
    * Realiza el seguimiento del progreso en tiempo real mediante barras de porcentaje.

## 💾 Salida de Datos
Los resultados se exportan como arreglos de NumPy (`.npy`) y archivos de etiquetas, listos para ser consumidos por el clasificador final:
* `train_embeddings.npy` / `train_labels.npy`
* `val_embeddings.npy` / `val_labels.npy`
* `test_embeddings.npy` / `test_labels.npy`

Route configuration

In [1]:
INPUT_CSV = "/content/drive/MyDrive/Experimentos/Data_Preparation/Data/csv/Dataset_Consolidation"
OUTPUT_EMBEDDINGS = "/content/drive/MyDrive/Experimentos/ProtT5/Embeddings/Generate"

Installation of libraries

In [2]:
!pip install sentencepiece

In [3]:
!pip install protobuf

Import of libraries

In [4]:
from transformers import AutoTokenizer, AutoModel, T5EncoderModel
import torch
import re
import gc
import numpy as np
import pandas as pd
import os
from tqdm import tqdm

Data loading

In [5]:
df_train = pd.read_csv(os.path.join(INPUT_CSV, "train.csv"))
df_val = pd.read_csv(os.path.join(INPUT_CSV, "val.csv"))
df_test = pd.read_csv(os.path.join(INPUT_CSV, "test.csv"))

train = []
train_ids = []
val = []
val_ids = []
test = []
test_ids = []

In [6]:
df_train.columns

Index(['Unnamed: 0', 'id', 'sequence', 'label'], dtype='object')

Format datasets

In [21]:
for idx, row in df_train.iterrows():
  train.append(row['sequence'])
  train_ids.append(row['id'])

for idx, row in df_val.iterrows():
  val.append(row['sequence'])
  val_ids.append(row['id'])

for idx, row in df_test.iterrows():
  test.append(row['sequence'])
  test_ids.append(row['id'])

ProtT5 model loading

In [19]:
MODEL_NAME = "Rostlab/prot_t5_xl_half_uniref50-enc"

os.makedirs(OUTPUT_EMBEDDINGS, exist_ok=True)

BATCH_SIZE = 8          # lower if OOM: 8 -> 4 -> 2 -> 1
MAX_LEN = 1024          # lower if OOM: 1024 -> 800 -> 512
USE_FP16 = True         # helps on CUDA
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

Preprocessing function

In [9]:
def preprocess_sequence(seq: str) -> str:
    """
    Rostlab models expect:
    - uppercase amino acids
    - uncommon residues replaced: U,Z,O,B -> X
    - spaces between residues
    """
    seq = str(seq).strip().upper()
    seq = re.sub(r"[UZOB]", "X", seq)
    return " ".join(list(seq))

Model and tokenizer loading

In [10]:
def load_model_and_tokenizer(model_name: str):
    tokenizer = AutoTokenizer.from_pretrained(model_name, do_lower_case=False)

    if "t5" in model_name.lower():
        model = T5EncoderModel.from_pretrained(model_name)
    else:
        model = AutoModel.from_pretrained(model_name)

    model.to(DEVICE)
    model.eval()
    return tokenizer, model

Mean Pooling Function (Aggregation by average)

In [11]:
def mean_pool(last_hidden_state: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
    """
    last_hidden_state: [B, L, H]
    attention_mask:    [B, L]
    Returns sequence embedding [B, H] by averaging non-padding tokens.
    """
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    masked_hidden = last_hidden_state * mask
    summed = masked_hidden.sum(dim=1)
    counts = mask.sum(dim=1).clamp(min=1e-9)
    return summed / counts

Batch Embedding Generation Function

In [12]:
def embed_sequences_in_batches(
    sequences,
    seq_ids,
    tokenizer,
    model,
    batch_size=8,
    max_len=1024,
    use_fp16=True
):
    """
    Returns:
      all_ids: list[str]
      all_embeddings: np.ndarray shape [N, H]
    """
    processed = [preprocess_sequence(s) for s in sequences]
    all_embeddings = []
    all_ids = []

    with torch.inference_mode():
        for start in tqdm(range(0, len(processed), batch_size), desc="Embedding"):
            end = min(start + batch_size, len(processed))
            batch_seqs = processed[start:end]
            batch_ids = seq_ids[start:end]

            encoded = tokenizer(
                batch_seqs,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=max_len
            )

            input_ids = encoded["input_ids"].to(DEVICE)
            attention_mask = encoded["attention_mask"].to(DEVICE)

            if DEVICE == "cuda" and use_fp16:
                with torch.autocast(device_type="cuda", dtype=torch.float16):
                    outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            else:
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)

            batch_emb = mean_pool(outputs.last_hidden_state, attention_mask)  # [B,H]
            batch_emb = batch_emb.detach().float().cpu().numpy()

            all_embeddings.append(batch_emb)
            all_ids.extend(batch_ids)

            # Memory cleanup
            del encoded, input_ids, attention_mask, outputs, batch_emb
            if DEVICE == "cuda":
                torch.cuda.empty_cache()
            gc.collect()

    all_embeddings = np.concatenate(all_embeddings, axis=0) if all_embeddings else np.empty((0, 0), dtype=np.float32)
    return all_ids, all_embeddings

Save function in CSV format

In [20]:
def save_embeddings_csv(split_name, all_ids, all_embeddings, output_dir):
    """
    Save exactly in requested format:
    seq_id | embedding(list)
    """
    # Convert embeddings to a numpy array
    all_embeddings = np.array(all_embeddings, dtype=np.float32)

    df_embeddings = pd.DataFrame({
        "seq_id": all_ids,
        "embedding": [emb.tolist() for emb in all_embeddings]
    })

    # Save
    output_path = f"{output_dir}/{split_name}_embeddings.csv"
    df_embeddings.to_csv(output_path, index=False)
    print(f"✅ Saved: {output_path} → {df_embeddings.shape}")

Execution: Model loading

In [14]:
print(f"Using device: {DEVICE}")
print(f"Loading model: {MODEL_NAME}")
tokenizer, model = load_model_and_tokenizer(MODEL_NAME)

Using device: cuda
Loading model: Rostlab/prot_t5_xl_half_uniref50-enc


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/656 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/238k [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/2.42G [00:00<?, ?B/s]

The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


TRAIN

In [15]:
print("Processing TRAIN...")
train_ids_out, train_emb = embed_sequences_in_batches(
    train, train_ids, tokenizer, model, BATCH_SIZE, MAX_LEN, USE_FP16
)
save_embeddings_csv("train", train_ids_out, train_emb, OUTPUT_EMBEDDINGS)

Processing TRAIN...


Embedding: 100%|██████████| 2444/2444 [14:29<00:00,  2.81it/s]


✅ Guardado: /content/drive/MyDrive/Experimentos/ProtT5/Embeddings/Generate/train_embeddings.csv → (19548, 2)


VAL

In [16]:
print("Processing VAL...")
val_ids_out, val_emb = embed_sequences_in_batches(
    val, val_ids, tokenizer, model, BATCH_SIZE, MAX_LEN, USE_FP16
)
save_embeddings_csv("val", val_ids_out, val_emb, OUTPUT_EMBEDDINGS)

Processing VAL...


Embedding: 100%|██████████| 641/641 [03:52<00:00,  2.76it/s]


✅ Guardado: /content/drive/MyDrive/Experimentos/ProtT5/Embeddings/Generate/val_embeddings.csv → (5125, 2)


TEST

In [17]:
print("Processing TEST...")
test_ids_out, test_emb = embed_sequences_in_batches(
    test, test_ids, tokenizer, model, BATCH_SIZE, MAX_LEN, USE_FP16
)
save_embeddings_csv("test", test_ids_out, test_emb, OUTPUT_EMBEDDINGS)

Processing TEST...


Embedding: 100%|██████████| 1961/1961 [11:52<00:00,  2.75it/s]


✅ Guardado: /content/drive/MyDrive/Experimentos/ProtT5/Embeddings/Generate/test_embeddings.csv → (15685, 2)
